In [2]:
import os
import glob
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

# Ensure TensorFlow detects the GPU
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"Using GPU: {physical_devices[0]}")
else:
    print("No GPU detected, falling back to CPU.")

I0000 00:00:1787189311.035441   16851 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [3]:
def resize_image(image, size=(256, 256)):
    """Resizes image or mask to target dimensions."""
    # Use INTER_NEAREST for masks (2D) and INTER_AREA for RGB images (3D)
    interpolation = cv2.INTER_AREA if len(image.shape) == 3 else cv2.INTER_NEAREST
    return cv2.resize(image, size, interpolation=interpolation)

def apply_clahe(image):
    """Applies Contrast Limited Adaptive Histogram Equalization (CLAHE)."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def apply_sharpening(image):
    """Applies a sharpening filter."""
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]])
    return cv2.filter2D(image, -1, kernel)

def apply_gaussian_filter(image, kernel_size=(3, 3), sigma=0):
    """Applies Gaussian Blur for noise suppression."""
    return cv2.GaussianBlur(image, kernel_size, sigma)

def apply_erosion(mask, kernel_size=(3, 3)):
    """Applies morphological erosion to mask boundaries."""
    kernel = np.ones(kernel_size, np.uint8)
    return cv2.erode(mask, kernel, iterations=1)

def apply_augmentations(image, mask_healthy, mask_polyp):
    """Applies synchronized spatial data augmentations."""
    if np.random.rand() > 0.5:
        # Horizontal Flip
        image = cv2.flip(image, 1)
        mask_healthy = cv2.flip(mask_healthy, 1)
        mask_polyp = cv2.flip(mask_polyp, 1)
        
    if np.random.rand() > 0.5:
        # Vertical Flip
        image = cv2.flip(image, 0)
        mask_healthy = cv2.flip(mask_healthy, 0)
        mask_polyp = cv2.flip(mask_polyp, 0)
        
    return image, mask_healthy, mask_polyp

In [4]:
class KvasirDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, img_paths, img_dir, healthy_dir, polyp_dir, batch_size=4, img_size=(256, 256), is_train=True):
        self.img_paths = img_paths
        self.img_dir = img_dir
        self.healthy_dir = healthy_dir
        self.polyp_dir = polyp_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self):
        return int(np.ceil(len(self.img_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_paths = self.img_paths[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        batch_x = []
        batch_y = []
        
        for img_path in batch_paths:
            filename = os.path.basename(img_path)
            healthy_path = os.path.join(self.healthy_dir, filename)
            polyp_path = os.path.join(self.polyp_dir, filename)
            
            # Read inputs
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            healthy_mask = cv2.imread(healthy_path, cv2.IMREAD_GRAYSCALE)
            polyp_mask = cv2.imread(polyp_path, cv2.IMREAD_GRAYSCALE)
            
            # 1. Resize
            image = resize_image(image, self.img_size)
            healthy_mask = resize_image(healthy_mask, self.img_size)
            polyp_mask = resize_image(polyp_mask, self.img_size)
            
            # 2. Image Preprocessing
            image = apply_clahe(image)
            image = apply_sharpening(image)
            image = apply_gaussian_filter(image)
            
            # 3. Mask Preprocessing
            healthy_mask = apply_erosion(healthy_mask)
            polyp_mask = apply_erosion(polyp_mask)
            
            # 4. Augmentations
            if self.is_train:
                image, healthy_mask, polyp_mask = apply_augmentations(image, healthy_mask, polyp_mask)
                
            # Normalize image
            image = image.astype(np.float32) / 255.0
            
            # Binarize masks
            healthy_mask = (healthy_mask > 127).astype(np.float32)
            polyp_mask = (polyp_mask > 127).astype(np.float32)
            
            # Stack into channels (H, W, 2)
            masks = np.stack([healthy_mask, polyp_mask], axis=-1)
            
            batch_x.append(image)
            batch_y.append(masks)
            
        return np.array(batch_x), np.array(batch_y)

In [6]:
# --- Squeeze-and-Excitation Module ---
def squeeze_and_excitation(input_tensor, ratio=16):
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])

# --- Convolutional Block Attention Module (CBAM) ---
def cbam_block(input_tensor, ratio=16):
    filters = input_tensor.shape[-1]
    
    # Channel Attention
    shared_dense_1 = layers.Dense(filters // ratio, activation='relu', use_bias=False)
    shared_dense_2 = layers.Dense(filters, use_bias=False)
    
    avg_pool = layers.GlobalAveragePooling2D()(input_tensor)
    avg_pool = layers.Reshape((1, 1, filters))(avg_pool)
    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    
    max_pool = layers.GlobalMaxPooling2D()(input_tensor)
    max_pool = layers.Reshape((1, 1, filters))(max_pool)
    max_out = shared_dense_2(shared_dense_1(max_pool))
    
    channel_attention = layers.Activation('sigmoid')(layers.Add()([avg_out, max_out]))
    x = layers.Multiply()([input_tensor, channel_attention])
    
    # Spatial Attention (FIXED: Wrapped tf ops in Lambda layers)
    avg_pool_sp = layers.Lambda(lambda z: tf.reduce_mean(z, axis=-1, keepdims=True))(x)
    max_pool_sp = layers.Lambda(lambda z: tf.reduce_max(z, axis=-1, keepdims=True))(x)
    concat = layers.Concatenate(axis=-1)([avg_pool_sp, max_pool_sp])
    
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid', use_bias=False)(concat)
    return layers.Multiply()([x, spatial_attention])

# --- U-Net Blocks ---
def se_encoder_block(inputs, filters):
    x = layers.Conv2D(filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = squeeze_and_excitation(x)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p

def cbam_decoder_block(inputs, skip_features, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=(2, 2), padding="same")(inputs)
    x = layers.Concatenate()([x, skip_features])
    
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    
    x = cbam_block(x)
    return x

# --- Full Architecture ---
def build_modified_unet(input_shape=(256, 256, 3), num_classes=2):
    inputs = layers.Input(input_shape)

    # Encoder
    s1, p1 = se_encoder_block(inputs, 64)
    s2, p2 = se_encoder_block(p1, 128)
    s3, p3 = se_encoder_block(p2, 256)
    s4, p4 = se_encoder_block(p3, 512)

    # Bottleneck
    b1 = layers.Conv2D(1024, 3, padding="same")(p4)
    b1 = layers.BatchNormalization()(b1)
    b1 = layers.Activation("relu")(b1)
    b1 = layers.Conv2D(1024, 3, padding="same")(b1)
    b1 = layers.BatchNormalization()(b1)
    b1 = layers.Activation("relu")(b1)
    b1 = squeeze_and_excitation(b1)

    # Decoder
    d1 = cbam_decoder_block(b1, s4, 512)
    d2 = cbam_decoder_block(d1, s3, 256)
    d3 = cbam_decoder_block(d2, s2, 128)
    d4 = cbam_decoder_block(d3, s1, 64)

    # Output (Sigmoid for multi-label binary segmentation)
    outputs = layers.Conv2D(num_classes, 1, padding="same", activation="sigmoid")(d4)

    return models.Model(inputs, outputs, name="Modified_UNet")

model = build_modified_unet()

In [10]:
# --- Class-Specific Dice Coefficients ---
def polyp_dice_coef(y_true, y_pred, smooth=1e-6):
    """Calculates Dice specifically for the Polyp channel (Index 1)"""
    y_true_polyp = K.flatten(y_true[..., 1])
    y_pred_polyp = K.flatten(y_pred[..., 1])
    intersection = K.sum(y_true_polyp * y_pred_polyp)
    return (2. * intersection + smooth) / (K.sum(y_true_polyp) + K.sum(y_pred_polyp) + smooth)

def healthy_dice_coef(y_true, y_pred, smooth=1e-6):
    """Calculates Dice specifically for the Healthy channel (Index 0)"""
    y_true_healthy = K.flatten(y_true[..., 0])
    y_pred_healthy = K.flatten(y_pred[..., 0])
    intersection = K.sum(y_true_healthy * y_pred_healthy)
    return (2. * intersection + smooth) / (K.sum(y_true_healthy) + K.sum(y_pred_healthy) + smooth)

# --- Combined Loss ---
def dice_bce_loss(y_true, y_pred):
    # Standard BCE handles both channels well
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    
    # We want the loss to focus equally on both classes, so we average their specific Dice losses
    polyp_loss = 1 - polyp_dice_coef(y_true, y_pred)
    healthy_loss = 1 - healthy_dice_coef(y_true, y_pred)
    
    return bce + ((polyp_loss + healthy_loss) / 2.0)

# Compile Metrics (Track Polyp and Healthy separately)
metrics = [
    polyp_dice_coef,
    healthy_dice_coef,
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]

In [11]:
# --- Configuration ---
IMG_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/images"
HEALTHY_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/healthy-masks"
POLYP_DIR = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/Kvasir-SEG/masks"
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/best_tf_modified_unet.keras"

BATCH_SIZE = 4
EPOCHS = 100
LR = 1e-4

# --- Data Splitting (80% Train, 10% Val, 10% Test) ---
all_images = sorted(glob.glob(os.path.join(IMG_DIR, "*.[pj][pn][g]")))
np.random.shuffle(all_images)

total_len = len(all_images)
train_idx = int(0.8 * total_len)
val_idx = int(0.9 * total_len)

train_imgs = all_images[:train_idx]
val_imgs = all_images[train_idx:val_idx]
test_imgs = all_images[val_idx:]

print(f"Total Images: {total_len} | Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

# --- Generators ---
train_gen = KvasirDataGenerator(train_imgs, IMG_DIR, HEALTHY_DIR, POLYP_DIR, batch_size=BATCH_SIZE, is_train=True)
val_gen = KvasirDataGenerator(val_imgs, IMG_DIR, HEALTHY_DIR, POLYP_DIR, batch_size=BATCH_SIZE, is_train=False)
test_gen = KvasirDataGenerator(test_imgs, IMG_DIR, HEALTHY_DIR, POLYP_DIR, batch_size=BATCH_SIZE, is_train=False)

# --- Compilation and Callbacks ---
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
              loss=dice_bce_loss,
              metrics=metrics)

callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_polyp_dice_coef', mode='max', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_polyp_dice_coef', mode='max', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

# --- Execute Training Loop ---
print(f"Targeting >85% DSC. Starting training for {EPOCHS} epochs...")

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

# --- Evaluate on Test Data ---
print("\n--- Training Complete. Evaluating on Test Set ---")
model.load_weights(MODEL_SAVE_PATH) # Load the best weights saved during training
test_metrics = model.evaluate(test_gen, verbose=1)

Total Images: 1000 | Train: 800 | Val: 100 | Test: 100
Targeting >85% DSC. Starting training for 100 epochs...
Epoch 1/100


I0000 00:00:1787190348.181748   20093 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_85526__.285
I0000 00:00:1787190366.167588   20093 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion', 20 bytes spill stores, 20 bytes spill loads



200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.9356 - healthy_dice_coef: 0.9391 - loss: 0.3724 - polyp_dice_coef: 0.6748 - precision: 0.9290 - recall: 0.9413

I0000 00:00:1787190405.905518   20090 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_89153__.50



Epoch 1: val_polyp_dice_coef improved from None to 0.73187, saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/best_tf_modified_unet.keras

Epoch 1: finished saving model to /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/conference/best_tf_modified_unet.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 91s 273ms/step - accuracy: 0.9321 - healthy_dice_coef: 0.9362 - loss: 0.3756 - polyp_dice_coef: 0.6860 - precision: 0.9252 - recall: 0.9381 - val_accuracy: 0.9415 - val_healthy_dice_coef: 0.9487 - val_loss: 0.3207 - val_polyp_dice_coef: 0.7319 - val_precision: 0.9355 - val_recall: 0.9464 - learning_rate: 1.0000e-04
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy: 0.9323 - healthy_dice_coef: 0.9362 - loss: 0.3626 - polyp_dice_coef: 0.7138 - precision: 0.9266 - recall: 0.9369
Epoch 2: val_polyp_dice_coef did not improve from 0.73187
200/200 ━━━━━━━━━━━━━━━━━━━━ 42s 208ms/step - accuracy: 0.9390 - healthy_dice_coef: 0.

KeyboardInterrupt: 